# Институциональные факторы листинговых режимов

Notebook для визуализации и анализа результатов. Вычислительный код — в отдельных скриптах:
- `build_master_table.py` — формирование master-таблицы (value + year)
- `temporal_audit.py` — аудит временного покрытия
- `explore_data.py` — первичная разведка датасетов

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 200)

plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

DATA_DIR = r'D:\_workspace\deep-research-listing\03_data\institutional'
FIGURES_DIR = os.path.join(DATA_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

## 1. Master-таблица

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'master_factors.csv'), index_col='country')

val_cols = [c for c in df.columns if c.endswith('_val')]
year_cols = [c for c in df.columns if c.endswith('_year')]
trend_cols = [c for c in df.columns if c.endswith('_delta_5y')]

print(f'Shape: {df.shape}')
print(f'Value columns ({len(val_cols)}): {val_cols}')
print(f'Year columns ({len(year_cols)}): {year_cols}')
print(f'Trend columns ({len(trend_cols)}): {trend_cols}')

In [ ]:
# Значения всех факторов
df[val_cols].round(2)

In [ ]:
# За какой год каждое значение
df[year_cols]

## 2. Покрытие и пропуски

In [ ]:
coverage = pd.DataFrame({
    'factor': [c.replace('_val', '') for c in val_cols],
    'n_available': [df[c].notna().sum() for c in val_cols],
    'n_missing': [df[c].isna().sum() for c in val_cols],
})
coverage['pct'] = (coverage['n_available'] / 48 * 100).round(1)
print(coverage.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 14))
mask = df[val_cols].isna().astype(int)
mask.columns = [c.replace('_val', '') for c in val_cols]
sns.heatmap(mask, cmap=['#2ecc71', '#e74c3c'],
            cbar_kws={'label': 'green=data, red=missing'},
            yticklabels=True, xticklabels=True, ax=ax)
ax.set_title('Data availability: 48 jurisdictions x 12 factors', fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'missing_data_heatmap.png'), dpi=150)
plt.show()

## 3. Temporal spread (WDI factors)

In [ ]:
wdi_factors = ['F7_mktcap_gdp', 'F7x_mktcap_usd', 'F7x_listed_n', 'Fx_savings_gdp', 'Fx_savings_usd']
for f in wdi_factors:
    yc = f'{f}_year'
    if yc in df.columns:
        years = df[yc].dropna()
        print(f'{f:25s} | range: {int(years.min())}-{int(years.max())} | '
              f'median: {int(years.median())} | '
              f'countries at median+: {(years >= years.median()).sum()}/48')

In [ ]:
# Visualize year distribution for F7_mktcap_gdp
fig, ax = plt.subplots(figsize=(10, 6))
years = df['F7_mktcap_gdp_year'].dropna().astype(int)
years.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Year of latest data')
ax.set_ylabel('Number of jurisdictions')
ax.set_title('F7 Market cap/GDP: year of latest available data per jurisdiction')
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'f7_year_distribution.png'), dpi=150)
plt.show()

## 4. Descriptive statistics

In [ ]:
df[val_cols].describe().round(3)

In [ ]:
# Distributions with Russia marked
n = len(val_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(val_cols):
    ax = axes[i]
    vals = df[col].dropna()
    ax.hist(vals, bins=15, edgecolor='black', alpha=0.7, color='steelblue')
    
    ru = df.loc['Russia', col] if pd.notna(df.loc['Russia', col]) else None
    if ru is not None:
        ax.axvline(ru, color='red', linewidth=2, linestyle='--', label=f'RU={ru:.1f}')
        ax.legend(fontsize=8)
    
    ax.set_title(col.replace('_val', ''), fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distributions (red = Russia)', fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'distributions.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation matrix

In [ ]:
corr = df[val_cols].corr()
corr.columns = [c.replace('_val', '') for c in corr.columns]
corr.index = [c.replace('_val', '') for c in corr.index]

fig, ax = plt.subplots(figsize=(12, 10))
mask_upper = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask_upper, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Factor correlation matrix', fontsize=14)
plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'correlation_matrix.png'), dpi=150)
plt.show()

## 6. WGI trend (2019 -> 2024)

In [ ]:
for t in trend_cols:
    factor = t.replace('_delta_5y', '')
    series = df[t].dropna().sort_values()
    
    fig, ax = plt.subplots(figsize=(10, 12))
    colors = ['red' if c == 'Russia' else ('orange' if df.loc[c, 'market_group'] == 'EM' else 'steelblue')
              for c in series.index]
    ax.barh(series.index, series.values, color=colors, edgecolor='black', linewidth=0.3)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Change (2019 -> 2024)')
    ax.set_title(f'{factor}: 5-year change\n(blue=DM, orange=EM, red=Russia)', fontsize=12)
    plt.tight_layout()
    fig.savefig(os.path.join(FIGURES_DIR, f'{factor}_trend_5y.png'), dpi=150)
    plt.show()

## 7. Russia position

In [ ]:
percentiles = {}
for col in val_cols:
    vals = df[col].dropna()
    ru = df.loc['Russia', col]
    if pd.notna(ru):
        pct = (vals < ru).sum() / len(vals) * 100
        percentiles[col.replace('_val', '')] = {'value': round(ru, 2), 'percentile': round(pct, 1)}
    else:
        percentiles[col.replace('_val', '')] = {'value': None, 'percentile': None}

pct_df = pd.DataFrame(percentiles).T
print('Russia position (percentile among 48 jurisdictions):')
print(pct_df)